In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/requirements.txt
/kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/selected-contexts.json
/kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/Scoring-Program-Task-LegalQA/metadata.yaml
/kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/Scoring-Program-Task-LegalQA/scoring.py
/kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/Scoring-Program-Task-LegalQA/rouge_score/tokenize.py
/kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/Scoring-Program-Task-LegalQA/rouge_score/rouge.py
/kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/Scoring-Program-Task-LegalQA/rouge_score/io.py
/kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/Scoring-Program-Task-LegalQA/rouge_score/tokenize_test.py
/kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/Scoring-Program-Task-LegalQA/rouge_score/rouge_s

# EDA

# DEPENDENCIES

In [2]:
!pip install -q -r /kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.5/250.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 73.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 41.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bi

# INDEXING

## Pre-loader

In [3]:
import json
from pathlib import Path

In [4]:
folder = Path('/kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/LegalQA - Public Test-20260827T072335Z-1-001/LegalQA - Public Test/selected-contexts')
docs = {}

for file in folder.rglob('*.json'):
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    docs[data['id']] = {
        'link': data.get('link'),
        'name': data.get('name'),
        'passage': data.get('passage'),
        'id': data['id'],
    }
    
with open('/kaggle/working/selected-contexts.json', 'w', encoding='utf-8') as f:
    json.dump(docs, f, ensure_ascii=False)

## Loader

In [5]:
import json
from langchain_core.documents import Document

In [6]:
def load_questions():
    with open('/kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/LegalQA - Public Test-20260827T072335Z-1-001/LegalQA - Public Test/public-official.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    questions = []
    
    for qid, item in data.items():
        questions.append(
            Document(
                page_content=item['question'],
                metadata={
                    'id': qid,
                },
                id=qid
            )
        )
        
    return questions

In [7]:
def load_answers():
    with open('/kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/selected-contexts.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    answers = []
    
    for pid, item in data.items():
        answers.append(
            Document(
                page_content=item['passage'],
                metadata={
                    'link': f"{item['link']}",
                    'name': f"{item['name']}",
                    'id': pid,
                },
                id=pid
            )
        )
        
    return answers

## Chunkings

In [8]:
import re
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_core.embeddings import Embeddings
# from langchain_experimental.text_splitter import SemanticChunker
# from FlagEmbedding import BGEM3FlagModel

In [9]:
# class bge_adapter(Embeddings):
#     def __init__(self, model=None):
#         self.model = model or BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
        
#     def embed_documents(self, texts):
#         return self.model.encode(texts, max_length=512)['dense_vecs'].tolist()
        
#     def embed_query(self, text):
#         return self.model.encode([text], max_length=512)['dense_vecs'][0].tolist()

In [10]:
def split_doc(doc):
    text = doc.page_content
    parts = re.split(r'(?=(?:^|\n)\s*Điều\s+\d+[a-zđ]?\s*(?:[.:]|\n))', text)
    docs = []
    
    for part in parts:
        part = part.strip()
        if not part:
            continue
            
        match = re.match(r'Điều\s+(\d+)', part)
        article = match.group(1) if match else None
        
        docs.append(
            Document(
                page_content=part,
                metadata={
                    **doc.metadata,
                    'article': article,
                }
            )
        )
        
    return docs if docs else [doc]

In [11]:
def chunking():
    docs = load_answers()
    articles = []
    
    for d in docs:
        articles.extend(split_doc(d))
        
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1200,
        chunk_overlap=200,
        strip_whitespace=True,
        length_function=len,
        separators=['\n\nKhoản ', '\n\n', '\n', '. ', ' ', '',],
    )
    
    chunks = text_splitter.split_documents(articles)
    return chunks

In [12]:
# def chunking():
#     docs = load_answers()
#     articles = []

#     for d in docs:
#         articles.extend(split_doc(d))

#     text_splitter = RecursiveCharacterTextSplitter(
#         chunk_size=1200,
#         chunk_overlap=200,
#         strip_whitespace=True,
#         length_function=len,
#         separators=['\n\nKhoản ', '\n\n', '\n', '. ', ' ', '',],
#     )

#     CHUNK_SIZE_LIMIT = 1200
#     # SEMANTIC_THRESHOLD = 3000
#     # semantic_splitter = None
#     chunks = []

#     for a in articles:
#         if len(a.page_content) <= CHUNK_SIZE_LIMIT:
#             chunks.append(a)
#         # elif len(a.page_content) <= SEMANTIC_THRESHOLD:
#         #     chunks.extend(text_splitter.split_documents([a]))
#         else:
#             chunks.extend(text_splitter.split_documents([a]))
#             # if semantic_splitter is None:
#             #     embeddings = bge_adapter()
#             #     semantic_splitter = SemanticChunker(
#             #         embeddings,
#             #         breakpoint_threshold_type='percentile',
#             #         breakpoint_threshold_amount=90,
#             #     )

#             # try:
#                 # sub_docs = semantic_splitter.create_documents(
#                 #     [a.page_content],
#                 #     metadatas=[a.metadata],
#                 # )
#                 # sub_docs = text_splitter.split_documents([a])
#             # except Exception:
#             #     sub_docs = [a]

#             # for sd in sub_docs:
#             #     if len(sd.page_content) <= CHUNK_SIZE_LIMIT:
#             #         chunks.append(sd)
#             #     else:
#             #         chunks.extend(text_splitter.split_documents([sd]))

#     chunks = [c for c in chunks if len(c.page_content.strip()) >= 30]
#     return chunks

In [13]:
!mkdir -p /kaggle/working/faiss_index

In [14]:
import os
import pickle

In [15]:
INDEX_DIR = '/kaggle/working/faiss_index'
CHUNKS_PATH = os.path.join(INDEX_DIR, 'chunks.pkl')

In [16]:
chunks = chunking()
os.makedirs(INDEX_DIR, exist_ok=True)

with open(CHUNKS_PATH, 'wb') as f:
    pickle.dump(chunks, f)

## Embeddings

In [17]:
from FlagEmbedding import BGEM3FlagModel

In [18]:
def bge_m3():
    return BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

In [19]:
def embeddings():
    chunks = chunking()
    model = bge_m3()
    answer_sentences = [chunk.page_content for chunk in chunks]
    
    vectors = model.encode(
        answer_sentences,
        batch_size=32,
        max_length=8192,
        )['dense_vecs']
    
    return vectors

In [20]:
import os
import numpy as np

In [21]:
INDEX_DIR = '/kaggle/working/faiss_index'
EMBEDDS_PATH = os.path.join(INDEX_DIR, 'embedds.npy')

In [22]:
vectors = embeddings()
os.makedirs(INDEX_DIR, exist_ok=True)

np.save(EMBEDDS_PATH, vectors)

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


initial target device: 100%|██████████| 2/2 [00:22<00:00, 11.04s/it]

Inference Embeddings: 100%|██████████| 6560/6560 [36:52<00:00,  2.96it/s]

Chunks: 100%|██████████| 2/2 [39:06<00:00, 1173.33s/it]


## Vector Store

In [23]:
import os
import pickle
import numpy as np
import faiss

In [24]:
INDEX_DIR = '/kaggle/working/faiss_index'
EMBEDDS_PATH = os.path.join(INDEX_DIR, 'embedds.npy')
INDEX_PATH = os.path.join(INDEX_DIR, 'index.faiss')

In [25]:
def vectorstore(vectors):
    idx = faiss.IndexFlatIP(1024)
    os.makedirs(INDEX_DIR, exist_ok=True)
    
    vectors = vectors.astype('float32')
    faiss.normalize_L2(vectors)
    
    idx.add(vectors)
    faiss.write_index(idx, INDEX_PATH)
    
    return idx

In [26]:
vectors = np.load(EMBEDDS_PATH)
idx = vectorstore(vectors)

# QUERY

## Retriever

In [27]:
import faiss
import numpy as np

In [28]:
INDEX_DIR = '/kaggle/working/faiss_index'
CHUNKS_PATH = os.path.join(INDEX_DIR, 'chunks.pkl')

In [29]:
idx = faiss.read_index(INDEX_PATH)

In [30]:
def get_chunks():
    with open(CHUNKS_PATH, 'rb') as f:
        return pickle.load(f)

In [31]:
chunks = get_chunks()
model = bge_m3()

def retriever(idx, chunks, model):
    def retrieve(question, k=20, top_n=5, use_rerank=True):
        q_vector = model.encode(
            [question],
            max_length=8192
            )['dense_vecs']
        q_vector = np.array(q_vector).astype('float32')
        faiss.normalize_L2(q_vector)
        
        scores, indices = idx.search(q_vector, k)
        candidates = [chunks[i] for i in indices[0] if i != -1]
        
        if not use_rerank:
            return candidates[:top_n]
        return rerank(question, candidates, top_n=top_n)
        
    def retrieve_batch(questions, k=20, top_n=5, batch_size=32, use_rerank=True):
        q_vectors = model.encode(
            questions,
            batch_size=batch_size,
            max_length=8192
            )['dense_vecs']
        q_vectors = np.array(q_vectors).astype('float32')
        faiss.normalize_L2(q_vectors)
        
        scores, indices = idx.search(q_vectors, k)
        all_candidates = []
        
        for i in range(len(questions)):
            candidates = [chunks[j] for j in indices[i] if j != -1]
            all_candidates.append(candidates)
            
        if not use_rerank:
            return [c[:top_n] for c in all_candidates]
            
        return rerank_batch(questions, all_candidates, top_n=top_n)
        
    return retrieve, retrieve_batch

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## Re-ranker

In [32]:
from sentence_transformers import CrossEncoder

In [33]:
_reranker = None

def get_reranker():
    global _reranker
    if _reranker is None:
        _reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')
    return _reranker
    
def rerank(question, candidates, top_n=5):
    if not candidates:
        return []
        
    reranker = get_reranker()
    pairs = [[question, c.page_content] for c in candidates]
    scores = reranker.predict(pairs)
    scored = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    
    return [doc for doc, score in scored[:top_n]]

def rerank_batch(questions, candidates_list, top_n=5):
    reranker = get_reranker()
    results = []
    
    for question, candidates in zip(questions, candidates_list):
        if not candidates:
            results.append([])
            continue
            
        pairs = [[question, c.page_content] for c in candidates]
        scores = reranker.predict(pairs)
        scored = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
        results.append([doc for doc, score in scored[:top_n]])
        
    return results

## Contexts

In [34]:
import os
import json

In [35]:
CONTEXTS_PATH = '/kaggle/working/contexts.json'

def contexts(output_path=CONTEXTS_PATH, k=20, top_n=5, use_rerank=True, batch_size=32):
    questions = load_questions()
    existing = {}
    
    if os.path.exists(output_path):
        with open(output_path, 'r', encoding='utf-8') as f:
            existing = json.load(f)
            
    pending = [q for q in questions if str(q.metadata.get('id')) not in existing]
    
    if not pending:
        print('Đã retrieve đủ context cho toàn bộ câu hỏi.')
        return
        
    print(f'Cần retrieve context cho {len(pending)}/{len(questions)} câu hỏi.')

    p_ids = [str(p.metadata.get('id')) for p in pending]
    p_questions = [p.page_content.strip() for p in pending]
    p_questions_search = [q.lower() for q in p_questions]

    chunks = get_chunks()
    model = bge_m3()
    
    r, r_batch = retriever(idx, chunks, model)
    
    for i in range(0, len(p_questions), batch_size):
        b_ids = p_ids[i: i + batch_size]
        b_questions = p_questions[i: i + batch_size]
        b_questions_search = p_questions_search[i: i + batch_size]
        b_docs = r_batch(b_questions_search, k=k, top_n=top_n, use_rerank=use_rerank)

        for qid, question, docs in zip(b_ids, b_questions, b_docs):
            context = '\n\n'.join(d.page_content for d in docs)
            existing[qid] = {
                'question': question,
                'context': context,
                }

        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(existing, f, ensure_ascii=False)

        print(f'Đã retrieve {min(i + batch_size, len(p_questions))}/{len(p_questions)}')
    print(f'\nHoàn tất retrieve. Đã lưu context cho {len(existing)}/{len(questions)} câu vào {output_path}.')

if __name__=='__main__':
    contexts()

Cần retrieve context cho 1000/1000 câu hỏi.


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.37it/s]


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

...

Đã retrieve 960/1000


Chunks: 100%|██████████| 2/2 [00:00<00:00, 30.57it/s]


Đã retrieve 992/1000


Chunks: 100%|██████████| 2/2 [00:00<00:00, 28.65it/s]


Đã retrieve 1000/1000

Hoàn tất retrieve. Đã lưu context cho 1000/1000 câu vào /kaggle/working/contexts.json.


## LLM

In [36]:
import gc
import torch

def free_gpu(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()

In [37]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [38]:
# SYSTEM_PROMPT = (
#     'You are a strict, citation-focused assistant for a private knowledge base.\n'
#     'RULES:\n'
#     '1. Use ONLY the provided context to answer.\n'
#     '2. If the answer is not clearly contained in the context, respond with EXACTLY '
#     "this sentence and nothing else: \"I don't know based on the provided documents.\"\n"
#     '3. Do NOT use outside knowledge, guessing, or web information.\n'
#     '4. Answer ONLY in Vietnamese. Do not include any English text, English preamble, '
#     'or English framing sentences (e.g. do NOT write "The answer is", "Based on the context", '
#     '"The provided text states" or similar). Do NOT repeat or restate the question.\n'
#     '5. Output ONLY the final Vietnamese answer text directly — no introduction, '
#     'no meta-commentary, no explanation of what you are doing.\n'
#     '6. When the context contains legal article/clause numbers (e.g. "Điều 5", "Khoản 2"), '
#     'cite them naturally in your answer, e.g. "Theo Điều 5 Khoản 2..." — matching the '
#     'legal writing style found in the context, not a casual paraphrase.\n'
#     '7. Provide a complete answer including relevant legal basis from the context, '
#     'not just a bare fact — unless the question explicitly asks for a single short value.\n\n'
# )

In [39]:
SYSTEM_PROMPT = (
    'Bạn là một trợ lý nghiêm ngặt, tập trung vào việc trích dẫn, dành cho một cơ sở kiến thức riêng tư.\n'
    'QUY TẮC:\n'
    '1. CHỈ sử dụng ngữ cảnh được cung cấp để trả lời.\n'
    '2. Nếu câu trả lời không được nêu rõ ràng trong ngữ cảnh, hãy trả lời CHÍNH XÁC câu này và không thêm bất kỳ nội dung nào khác: "Tôi không biết dựa trên các tài liệu được cung cấp."\n'
    '3. KHÔNG sử dụng kiến thức bên ngoài, suy đoán hoặc thông tin từ web.\n'
    '4. CHỈ trả lời bằng tiếng Việt. Không được bao gồm bất kỳ nội dung tiếng Anh nào, phần mở đầu bằng tiếng Anh hoặc câu dẫn bằng tiếng Anh (ví dụ: KHÔNG viết "The answer is", "Based on the context", "The provided text states" hoặc các câu tương tự). Không được lặp lại hoặc diễn đạt lại câu hỏi.\n'
    '5. CHỈ xuất trực tiếp phần câu trả lời cuối cùng bằng tiếng Việt — không có lời giới thiệu, không có bình luận mang tính meta, không giải thích về những gì bạn đang làm.\n'
    '6. Khi ngữ cảnh có chứa số điều/khoản của văn bản pháp luật (ví dụ: "Điều 5", "Khoản 2"), hãy trích dẫn chúng một cách tự nhiên trong câu trả lời, ví dụ: "Theo Điều 5 Khoản 2..." — phù hợp với văn phong pháp lý được sử dụng trong ngữ cảnh, không diễn đạt lại theo cách nói thông thường.\n'
    '7. Cung cấp câu trả lời đầy đủ, bao gồm căn cứ pháp lý có liên quan từ ngữ cảnh, không chỉ đưa ra một thông tin đơn lẻ — trừ khi câu hỏi yêu cầu rõ ràng một giá trị ngắn duy nhất.\n\n'
)

In [40]:
def load_model(device_map=None):
    model_name = "Qwen/Qwen2.5-3B-Instruct"
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map=device_map or 'auto',
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    )
    model.eval()
    
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer
    
def generate(model, tokenizer, question, context):
    answers = generate_batch(model, tokenizer, [question], [context])
    return answers[0]

@torch.no_grad()
def generate_batch(model, tokenizer, questions, contexts):
    messages_list = [
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {q}"},
        ]
        for q, ctx in zip(questions, contexts)
    ]
    texts = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in messages_list
    ]
    
    tokenizer.padding_side = "left"
    model_inputs = tokenizer(texts, return_tensors="pt", padding=True)
    model_inputs = {k: v.to(next(model.parameters()).device) for k, v in model_inputs.items()}
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    
    results = []
    
    for input_ids, output_ids in zip(model_inputs['input_ids'], generated_ids):
        new_tokens = output_ids[len(input_ids):]
        results.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
        
    return results

## Submission

In [41]:
# def rag_pipeline():
#     print('Đang load retriever và model...\n')
#     idx, chunks, model = vectorstore()
#     r, _ = retriever(idx, chunks, model)
#     model, tokenizer = load_model()

#     print('\nSẵn sàng, gõ "exit" để thoát chương trình.\n')

#     while True:
#         question = input('Question: ').strip().lower()
#         if question == 'exit':
#             break
#         print()

#         docs = r(question, k=20, top_n=5, use_rerank=True)
#         context = '\n\n'.join(d.page_content for d in docs)
#         answer = generate(model, tokenizer, question, context)
#         print(f'\nAnswer: {answer}\n')

# if __name__ == "__main__":
#     rag_pipeline()

In [42]:
import os
import json
import time

In [ ]:
CONTEXTS_PATH = '/kaggle/working/contexts.json'
BATCH_SIZE = 16

def load_existing_submission(path='/kaggle/working/submission.json'):
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

def make_submission(contexts_path=CONTEXTS_PATH, output_path='/kaggle/working/submission.json', batch_size=BATCH_SIZE, device_map=None):
    with open(contexts_path, 'r', encoding='utf-8') as f:
        contexts = json.load(f)
        
    submission = load_existing_submission(output_path)
    p_ids = [qid for qid in contexts if qid not in submission]
    
    if not p_ids:
        print('Không còn câu nào cần xử lý.')
        return

    print(f'Cần xử lý {len(p_ids)}/{len(contexts)} câu trả lời.')
    print(f'Batch size: {batch_size}')
    print()

    global _reranker
    free_gpu(_reranker) if '_reranker' in globals() and _reranker else None
    torch.cuda.empty_cache()
    
    model, tokenizer = load_model(device_map=device_map)
    start = time.time()
    total_done = 0

    for i in range(0, len(p_ids), batch_size):
        b_ids = p_ids[i: i + batch_size]
        b_questions = [contexts[qid]['question'] for qid in b_ids]
        b_contexts = [contexts[qid]['context'] for qid in b_ids]

        try:
            b_answers = generate_batch(model, tokenizer, b_questions, b_contexts)
        except torch.cuda.OutOfMemoryError as e:
            print(f'OOM batch {b_ids[0]}: {e}')
            
            torch.cuda.empty_cache()
            b_answers = []
            
            for q, c in zip(b_questions, b_contexts):
                try:
                    a = generate_batch(model, tokenizer, [q], [c])[0]
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    a = 'Tôi không biết dựa trên các tài liệu được cung cấp.'
                
                b_answers.append(a)

        for qid, answer in zip(b_ids, b_answers):
            submission[qid] = {'answer': answer}

        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(submission, f, ensure_ascii=False, indent=4)

        total_done += len(b_ids)
        elapsed = time.time() - start
        avg = elapsed / total_done
        remaining = avg * (len(p_ids) - total_done)
        print(
            f'Đã xử lý {total_done}/{len(p_ids)} '
            f'({b_ids[0]}...{b_ids[-1]}) — còn ~{remaining / 60:.1f} phút')
    print(f'\nHoàn tất. Tổng cộng {len(submission)}/{len(contexts)} câu đã xử lý.')

if __name__ == "__main__":
    make_submission()

Cần xử lý 1000/1000 câu trả lời.
Batch size: 16



config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


OOM batch 80189: CUDA out of memory. Tried to allocate 5.08 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.96 GiB is free. Including non-PyTorch memory, this process has 12.60 GiB memory in use. Of the allocated memory 12.20 GiB is allocated by PyTorch, and 283.15 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Đã xử lý 16/1000 (80189...13409) — còn ~230.2 phút
OOM batch 134019: CUDA out of memory. Tried to allocate 5.79 GiB. GPU 0 has a total capacity of 14.56 GiB of which 5.73 GiB is free. Including non-PyTorch memory, this process has 8.83 GiB memory in use. Of the allocated memory 7.89 GiB is allocated by PyTorch, and 837.02 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=e